In [ ]:
[finemap_DEN_SLA]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}_DEN_SLALOM.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container 
    ############################################################
    # 0. Load data
    ############################################################
    
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)
        # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                             full.names = TRUE, pattern = "\\.R$")) source(f)
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    path = ${_input:r}
    dat <- readRDS(path)
    
    
    n        <- dat$n_samples
    n_snps   <- dat$n_snp
    ture_vars = dat$true_vars
    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
    
      # Orient so that rows = B pseudo-individuals, columns = p variants
      if (ncol(U) == p) {
        Uv <- U            # already B x p  (U = W^T X form)
      } else if (nrow(U) == p) {
        Uv <- t(U)         # stored p x B -> transpose to B x p
      } else {
        stop(sprintf(
          "Sketch U (%d x %d) matches neither dimension to variant count p = %d",
          nrow(U), ncol(U), p))
      }
    
      R <- cor(Uv)                 # p x p, diagonal exactly 1
      R[is.na(R)] <- 0             # guard: zero-variance variant columns
      R
    }
    
    ############################################################
    # 1. Helper: RAISS imputation on outliers
    ############################################################
    
    run_raiss_on_outliers <- function(outlier_flags, z_vec, ref_df, LD_mat) {
    
      known_idx <- which(!outlier_flags)
    
    
      known_df <- data.frame(
        chrom      = ref_df$chrom[known_idx],
        pos        = ref_df$pos[known_idx],
        variant_id = ref_df$variant_id[known_idx],
        A1         = ref_df$A1[known_idx],
        A2         = ref_df$A2[known_idx],
        z          = z_vec[known_idx],
        stringsAsFactors = FALSE
      )
    
      result <- raiss(
        ref_panel     = ref_df,
        known_zscores = known_df,
        LD_matrix     = LD_mat,
        lamb = 0.01,
        rcond = 0.01,
        R2_threshold = 0.6,
        minimum_ld   = 5,
        verbose      = FALSE
      )
    
      combined_z <- z_vec
    
      if (!is.null(result) && !is.null(result$result_nofilter)) {
        raiss_df <- result$result_nofilter
    
        for (i in which(outlier_flags)) {
          vid <- ref_df$variant_id[i]
          ri  <- which(raiss_df$variant_id == vid)
    
          if (length(ri) == 1 && !is.na(raiss_df$z[ri])) {
            combined_z[i] <- raiss_df$z[ri]
          }
        }
      }
    
      list(z = combined_z, raiss_result = result)
    }
    
    
    
    ############################################################
    # 2. Run QC methods (DENTIST + SLALOM)
    ############################################################
    #  NOTE: z is now passed in explicitly so the SAME input (same LD,
    #  same variant annotations) can be run against different z vectors
    #  (original vs. flipped). ref_df is only used for variant
    #  annotations in RAISS, so it is identical across the two runs.
    
    finemap_QC <- function(input, z, n, name) {
          p  <- length(z)
    
          # LD reference: use provided LD, else derive it from the genotype sketch U
          if (!is.null(input$LD)) {
            LD <- input$LD
          } else if (!is.null(input$U)) {
            cat(sprintf("No LD in input; deriving R_hat = cor(U) from sketch (p = %d)\n", p))
            LD <- compute_LD_from_sketch(input$U, p)
          } else {
            stop("input must contain either 'LD' or 'U' (genotype sketch)")
          }
          ref_df   <- input$sumstat   # variant annotations (chrom/pos/variant_id/A1/A2)
    
    
                # DENTIST
        dentist_sw <- dentistSingleWindow(
          zScore  = z,
          R       = LD,
          nSample = n
        )
            dentist_outlier_flags <- dentist_sw$outlier
    
            # SLALOM
            slalom_result   <- slalom(zScore = z, R = LD)
            slalom_outliers <- slalom_result$data$outliers
            slalom_outliers[is.na(slalom_outliers)] <- FALSE
    
    
            ############################################################
            # 3. Run RAISS imputations
            ############################################################
    
            cat(sprintf("SLALOM+RAISS: imputing %d variants\n", sum(slalom_outliers)))
            slalom_raiss  <- run_raiss_on_outliers(slalom_outliers, z, ref_df, LD)
    
            cat(sprintf("DENTIST+RAISS: imputing %d variants\n", sum(dentist_outlier_flags)))
            dentist_raiss <- run_raiss_on_outliers(dentist_outlier_flags, z, ref_df, LD)
    
    
            ############################################################
            # 4. Construct QC-adjusted z-scores
            ############################################################
    
            # 4.1 DENTIST only
            dentist_qc_z <- z
            dentist_qc_z[dentist_outlier_flags] <- dentist_sw$imputed_z[dentist_outlier_flags]
    
            # 4.2 SLALOM + RAISS
            slalom_raiss_z <- slalom_raiss$z
    
            # 4.3 DENTIST + RAISS
            dentist_raiss_z <- dentist_raiss$z
    
    
            ############################################################
            # 5. Summary statistics
            ############################################################
    
            cat(sprintf("=== Three-Way Comparison (%s) ===\n", name))
            cat(sprintf("Total variants: %d\n\n", n_snps))
    
            cat("Outlier detection:\n")
            cat(sprintf("  DENTIST: %d (%.1f%%)\n",
                        sum(dentist_outlier_flags), 100 * mean(dentist_outlier_flags)))
    
            cat(sprintf("  SLALOM:  %d (%.1f%%)\n",
                        sum(slalom_outliers), 100 * mean(slalom_outliers)))
    
            cat(sprintf("  Overlap: %d\n\n",
                        sum(dentist_outlier_flags & slalom_outliers)))
    
    
    
    
            ############################################################
            # 7. SuSiE fine-mapping comparison
            ############################################################
    
            library(susieR)
    
            run_susie <- function(z_vec, LD, n) {
              fit <- susie_rss(
                z           = z_vec,
                R           = LD,
                n           = n,
                control = susie_rss_control(
                    check_prior = FALSE
                )
              )
              fit
            }
    
            # raw (no QC) SuSiE on the z passed in, for reference
            raw_res = run_susie(z, LD, n)
            raw_res$outlier_number = 0
    
            dentist_qc_res = run_susie(dentist_qc_z, LD, n)
            dentist_qc_res$outlier_number = sum(dentist_outlier_flags)
    
            dentist_raiss_res = run_susie(dentist_raiss_z, LD, n)
            dentist_raiss_res$outlier_number = sum(dentist_outlier_flags)
    
            slalom_raiss = run_susie(slalom_raiss_z, LD, n)
            print(sum(slalom_outliers) )
            slalom_raiss$outlier_number =  sum(slalom_outliers)
    
            output = list(raw_res = raw_res, dentist_qc_res = dentist_qc_res,
                          dentist_raiss_res = dentist_raiss_res, slalom_raiss = slalom_raiss,
                          input_path = path)
    
             return(output)
    }
    
    
    
        result_list <- list()
    
        # Same gtex_input (same LD, same annotations); only z differs.
        result_list[["gtex_original"]] <- finemap_QC(
            dat$gtex_input,
            z    = dat$gtex_input$sumstat_original$z,
            n    = n,
            name = "gtex_original")
    
        result_list[["gtex_flip"]] <- finemap_QC(
            dat$gtex_input,
            z    = dat$gtex_input$sumstat$z,
            n    = n,
            name = "gtex_flip")
    
        output = list(result_list = result_list, n_sample = n, input_path = path,
                  true_vars = dat$true_vars, n_snp = dat$n_snp,
                  miss_prop = dat$miss_prop, miss_fac = dat$miss_fac)
        saveRDS(output, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_DEN_Flip/"
mkdir -p ${work_dir}
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
cd ${work_dir}/code

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J Fine_Den
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/Fine_Den.%J.out
#BSUB -e /WORK_DIR/log/Fine_Den.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/03.Fig1B.Allele_Flip_Other_Methods.ipynb" finemap_DEN_SLA \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Allele_flip_Gtex_Simulation/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/
EOF

miss_fac=-1.0

base_sh="base_script"
for miss_prop in 0.001 0.01 0.1; do
    for causal in 1 2 3; do
       for h2g in 0.05 0.1 0.15; do
            output_script="miss_prop_${miss_prop}_miss_fac_${miss_fac}_causal_${causal}_h2g_${h2g}.sh"
            cat ${base_sh}| sed "s|MISSPROP|${miss_prop}|g" |sed "s|MISSFAC|${miss_fac}|g" |sed "s|CAUSAL|${causal}|g" |sed "s|WORK_DIR|${work_dir}|g" |sed "s|H2G|${h2g}|g"   > ${output_script}
            sb ${output_script}
        done
    done
done

In [ ]:
[finemap_CARMA]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container 
    
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)
    library(CARMA)
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
    
      # Orient so that rows = B pseudo-individuals, columns = p variants
      if (ncol(U) == p) {
        Uv <- U            # already B x p  (U = W^T X form)
      } else if (nrow(U) == p) {
        Uv <- t(U)         # stored p x B -> transpose to B x p
      } else {
        stop(sprintf(
          "Sketch U (%d x %d) matches neither dimension to variant count p = %d",
          nrow(U), ncol(U), p))
      }
    
      R <- cor(Uv)                 # p x p, diagonal exactly 1
      R[is.na(R)] <- 0             # guard: zero-variance variant columns
      R
    }
    
    path = ${_input:r}
    
    dat = readRDS(path)
    
    n <- dat$n_samples
    
    #  z is now passed in explicitly so the SAME input (same LD) can be
    #  run against different z vectors (original vs. flipped).
    finemap_CARMA <- function(input, z, n, name) {
      p  <- length(z)
      # LD reference: use provided LD, else derive it from the genotype sketch U
      if (!is.null(input$LD)) {
        LD <- input$LD
      } else if (!is.null(input$U)) {
        cat(sprintf("No LD in input; deriving R_hat = cor(U) from sketch (p = %d)\n", p))
        LD <- compute_LD_from_sketch(input$U, p)
      } else {
        stop("input must contain either 'LD' or 'U' (genotype sketch)")
      }
        
     z.list      <- list(z)
     ld.list     <- list(as.matrix(LD))
     lambda.list <- list(1)
    
     CARMA.results <- CARMA(
      z.list,
      ld.list,
      lambda.list = lambda.list,
      outlier.switch = TRUE
     )
        CARMA.results$sets = list()
        if(length(CARMA.results[[1]]$`Credible set`[[2]])!=0){
            real_set <- CARMA.results[[1]]$`Credible set`[[2]]
        }else{
            real_set = NULL
        }
        
        CARMA.results$sets$cs = real_set
      
        return(CARMA.results)
    }
    
    result_list <- list()
    
    # Same gtex_input (same LD); only z differs.
    result_list[["gtex_original"]] <- finemap_CARMA(
        dat$gtex_input,
        z    = dat$gtex_input$sumstat_original$z,
        n    = n,
        name = "gtex_original")
    
    result_list[["gtex_flip"]] <- finemap_CARMA(
        dat$gtex_input,
        z    = dat$gtex_input$sumstat$z,
        n    = n,
        name = "gtex_flip")
    
    output = list(result_list = result_list, n_sample = n, input_path = path, 
              true_vars = dat$true_vars, n_snp = dat$n_snp,
              miss_prop = dat$miss_prop, miss_fac = dat$miss_fac)
    saveRDS(output, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_CARMA_Flip/"
mkdir -p ${work_dir}
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
cd ${work_dir}/code

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J Fine_Den
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/Fine_Den.%J.out
#BSUB -e /WORK_DIR/log/Fine_Den.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/03.Fig1B.Allele_Flip_Other_Methods.ipynb" finemap_CARMA \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Allele_flip_Gtex_Simulation/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/
EOF

miss_fac=-1.0

base_sh="base_script"
for miss_prop in 0.001 0.01 0.1; do
    for causal in 1 2 3; do
       for h2g in 0.05 0.1 0.15; do
            output_script="miss_prop_${miss_prop}_miss_fac_${miss_fac}_causal_${causal}_h2g_${h2g}.sh"
            cat ${base_sh}| sed "s|MISSPROP|${miss_prop}|g" |sed "s|MISSFAC|${miss_fac}|g" |sed "s|CAUSAL|${causal}|g" |sed "s|WORK_DIR|${work_dir}|g" |sed "s|H2G|${h2g}|g"   > ${output_script}
            #sb ${output_script}
        done
    done
done

In [ ]:
[finemap_rsparse_pro]
parameter: genofile = paths
# pheno_file: give genotype file (in plink)，we can read the gentype matrix. These files are separated by TADs.
parameter: cwd = path("output")
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
# specify the number of causal variants
# specify the number of traits (phenotypes)
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rsparsepro.rds'
R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container 
    library(tidyverse)
    path = ${_input:r}
    dat  <- readRDS(path)
    compute_LD_from_sketch <- function(U, p) {
      U <- as.matrix(U)
    
      # Orient so that rows = B pseudo-individuals, columns = p variants
      if (ncol(U) == p) {
        Uv <- U            # already B x p  (U = W^T X form)
      } else if (nrow(U) == p) {
        Uv <- t(U)         # stored p x B -> transpose to B x p
      } else {
        stop(sprintf(
          "Sketch U (%d x %d) matches neither dimension to variant count p = %d",
          nrow(U), ncol(U), p))
      }
    
      R <- cor(Uv)                 # p x p, diagonal exactly 1
      R[is.na(R)] <- 0             # guard: zero-variance variant columns
      R
    }
    
    #  z is now passed in explicitly, and `tag` distinguishes the intermediate
    #  files so the two runs (original vs. flipped) don't clobber each other.
    finemap_Rsparse <- function(input,
                                z,
                                tag,
                                true_vars = NULL,
                                input_file = NULL,
                                python_bin = "python",
                                rsparsepro_script = "/sc/arion/projects/ad-omics/haochen/rss_qc/RSparsePro_LD/src/rsparsepro_ld.py") {
      p  <- length(z)
      # LD reference: use provided LD, else derive it from the genotype sketch U
      if (!is.null(input$LD)) {
        LD <- input$LD
      } else if (!is.null(input$U)) {
        cat(sprintf("No LD in input; deriving R_hat = cor(U) from sketch (p = %d)\n", p))
        LD <- compute_LD_from_sketch(input$U, p)
      } else {
        stop("input must contain either 'LD' or 'U' (genotype sketch)")
      }
        
      LD <- as.matrix(LD)
      rsid <- colnames(LD)
      if (is.null(rsid)) {
        rsid <- paste0("snp_", seq_along(z))
      }
    
      z_df <- tibble(
        RSID = rsid,
        Z = z
      )
    
      # Distinct on-disk prefix per run so original vs. flipped don't collide.
      out_prefix <- paste0(${_output:r}, ".", tag)
    
      z_file      <- paste0(out_prefix, ".z.txt")
      ld_file     <- paste0(out_prefix, ".ld.txt")
      result_path <- paste0(out_prefix, ".rsparsepro.txt")
      rds_path    <- paste0(out_prefix, ".rsparsepro.rds")
    
      write_tsv(z_df, z_file)
    
      write.table(
        LD,
        file = ld_file,
        sep = " ",
        row.names = FALSE,
        col.names = FALSE,
        quote = FALSE
      )
    
      cmd <- paste(
        shQuote(python_bin),
        shQuote(rsparsepro_script),
        "--z", shQuote(z_file),
        "--ld", shQuote(ld_file),
        "--save", shQuote(out_prefix)
      )
    
      status <- system(cmd)
    
      res <- read_tsv(result_path, show_col_types = FALSE) %>%
        mutate(index = row_number())
    
      if (!"cs" %in% names(res)) {
        stop("RSparsePro output does not contain column 'cs'.")
      }
    
      cs_index <- setdiff(unique(res$cs), 0)
    
      result_rds <- list()
      result_rds$fit <- res
      result_rds$set <- list()
    
      if (length(cs_index) == 0) {
        result_rds$set$cs <- NULL
      } else {
        cs <- vector("list", length(cs_index))
        for (i in seq_along(cs_index)) {
          cs[[i]] <- res %>%
            filter(cs == cs_index[i]) %>%
            pull(index)
        }
        result_rds$set$cs <- cs
      }
    
      return(result = list(fit = result_rds))
    }
    
    
    # Same gtex_input (same LD); only z differs.
    result_gtex_original <- finemap_Rsparse(
      input      = dat$gtex_input,
      z          = dat$gtex_input$sumstat_original$z,
      tag        = "gtex_original",
      true_vars  = dat$true_vars,
      input_file = path
    )
    
    result_gtex_flip <- finemap_Rsparse(
      input      = dat$gtex_input,
      z          = dat$gtex_input$sumstat$z,
      tag        = "gtex_flip",
      true_vars  = dat$true_vars,
      input_file = path
    )
    
    result_list <- list()
    
    result_list[["gtex_original"]] <- result_gtex_original
    result_list[["gtex_flip"]]     <- result_gtex_flip
    
    output = list(result_list = result_list, n_sample = dat$n_samples, input_path = path, 
              true_vars = dat$true_vars, n_snp = dat$n_snp,
              miss_prop = dat$miss_prop, miss_fac = dat$miss_fac)
    saveRDS(output, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_RSP_Flip/"
mkdir -p ${work_dir}
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
cd ${work_dir}/code

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J Fine_rsp
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/Fine_RSP.%J.out
#BSUB -e /WORK_DIR/log/Fine_RSP.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/03.Fig1B.Allele_Flip_Other_Methods.ipynb" finemap_rsparse_pro \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Allele_flip_Gtex_Simulation/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/miss_prop_MISSPROP_miss_fac_MISSFAC/
EOF

miss_fac=-1.0

base_sh="base_script"
for miss_prop in 0.001 0.01 0.1; do
    for causal in 1 2 3; do
       for h2g in 0.05 0.1 0.15; do
            output_script="miss_prop_${miss_prop}_miss_fac_${miss_fac}_causal_${causal}_h2g_${h2g}.sh"
            cat ${base_sh}| sed "s|MISSPROP|${miss_prop}|g" |sed "s|MISSFAC|${miss_fac}|g" |sed "s|CAUSAL|${causal}|g" |sed "s|WORK_DIR|${work_dir}|g" |sed "s|H2G|${h2g}|g"   > ${output_script}
            #sb ${output_script}
        done
    done
done

## Summary

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_DEN_Flip/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" DEN_SLA_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_CARMA_Flip/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" single_QC_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fine_RSP_Flip//"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/11.Fig3A.Other_Methods.ipynb" single_QC_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}